# Case Study 3: AI Customer Support Agent
## Multi-Agent Architecture with Google ADK + LangChain

This notebook demonstrates a multi-agent customer support system:

- **Planner Agent** — routes incoming requests to the right specialist
- **Knowledge Agent** — retrieves relevant docs via LangChain RAG (FAISS)
- **Diagnostic Agent** — runs automated system/account checks
- **Escalation Agent** — packages full context for human handoff

```
Customer Message
      |
  [Orchestrator] -- session state, execution flow
      |
  [Planner] -> [Knowledge] -> [Diagnostic] -> [Escalation]
                   |
              [FAISS Store]
```

In [1]:

def _clean(content):
    if isinstance(content, list):
        return "".join(b["text"] for b in content if isinstance(b, dict) and "text" in b)
    return str(content) if content else ""

import os
import json
import uuid
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

import os
os.environ["GOOGLE_API_KEY"] = "your-gemini-api-key-here"

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

print("Imports and LLM initialized.")

Imports and LLM initialized.


In [2]:
# Mock Data

KNOWLEDGE_BASE = [
    {"id": "KB-1042", "title": "Login Troubleshooting Guide",
     "content": "If you are unable to log in, try: 1) Clear browser cache and cookies. 2) Try incognito/private window. 3) Ensure correct login URL: app.example.com/login. 4) Check caps lock. 5) If 'Account Locked', wait 30 minutes. 6) Try a different browser. 7) Disable browser extensions."},
    {"id": "KB-1003", "title": "Password Reset Guide",
     "content": "To reset your password: 1) Go to app.example.com/login. 2) Click 'Forgot Password?'. 3) Enter your email. 4) Check inbox for reset link (check spam). 5) Click link and set new password. Link expires after 24 hours. Passwords must be 12+ chars with uppercase, number, and special character."},
    {"id": "KB-1015", "title": "Advanced Login Issues",
     "content": "If basic troubleshooting fails: 1) Try a different browser entirely. 2) Disable all extensions. 3) Check if other pages on app.example.com load. 4) Try different network (WiFi to mobile data). 5) Check browser console (F12) for errors. 6) Disconnect corporate proxy/VPN."},
    {"id": "KB-1050", "title": "Known Issues - June 2026",
     "content": "Current known issues: 1) Dashboard slow for 10,000+ record accounts - fix ETA July 5. 2) CSV export may timeout for 50,000+ rows - workaround: apply filters. 3) Safari 18 layout issues on settings page - use Chrome/Firefox."},
    {"id": "KB-1060", "title": "Billing and Subscription FAQ",
     "content": "Billing: 1) Invoices sent 1st of each month. 2) Update payment: Settings > Billing > Payment Methods. 3) Cancel: Settings > Billing > Cancel Subscription. 4) Refund requests: billing@example.com with invoice number."},
]

SUPPORT_TICKETS = [
    {"id": "TKT-4401", "subject": "Cannot log in after password change", "status": "resolved",
     "resolution": "Customer was using old bookmarked URL. Updated to correct URL."},
    {"id": "TKT-4422", "subject": "Login page blank on Safari", "status": "resolved",
     "resolution": "Known Safari 18 issue. Customer switched to Chrome."},
    {"id": "TKT-4455", "subject": "MFA code not arriving", "status": "open", "resolution": None},
    {"id": "TKT-4460", "subject": "Dashboard timeout for large accounts", "status": "in_progress",
     "resolution": "Engineering working on fix, ETA July 5."},
]

SYSTEM_HEALTH = {
    "auth_service": {"status": "healthy", "latency_ms": 45},
    "api_gateway": {"status": "healthy", "latency_ms": 32},
    "database": {"status": "healthy", "latency_ms": 12},
    "cdn": {"status": "healthy", "latency_ms": 8},
    "email_service": {"status": "degraded", "latency_ms": 2100, "note": "Delayed delivery, emails may take up to 10 minutes"},
}

print(f"Loaded {len(KNOWLEDGE_BASE)} KB articles, {len(SUPPORT_TICKETS)} tickets, {len(SYSTEM_HEALTH)} services.")

Loaded 5 KB articles, 4 tickets, 5 services.


In [3]:
# Simple keyword-based knowledge search (no external embedding API needed)
from difflib import SequenceMatcher

ALL_DOCS = [f"{a['title']}\n{a['content']}" for a in KNOWLEDGE_BASE]
for ticket in SUPPORT_TICKETS:
    if ticket["status"] == "resolved" and ticket["resolution"]:
        ALL_DOCS.append(f"Resolved: {ticket['subject']}\nResolution: {ticket['resolution']}")

def search_knowledge(query: str, k: int = 3) -> list[str]:
    query_lower = query.lower()
    scored = []
    for doc in ALL_DOCS:
        score = sum(1 for word in query_lower.split() if word in doc.lower())
        score += SequenceMatcher(None, query_lower, doc.lower()).ratio()
        scored.append((score, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored[:k]]

print(f"Knowledge search ready with {len(ALL_DOCS)} documents.")

Knowledge search ready with 7 documents.


In [4]:
# Specialist Agent Classes

class PlannerAgent:
    ROUTING_PROMPT = PromptTemplate(
        input_variables=["query", "history", "attempt_count"],
        template=(
            "You are a support request router. Classify the customer query and decide which agents to invoke.\n\n"
            "Customer query: {query}\nConversation history: {history}\nPrior attempts: {attempt_count}\n\n"
            "Respond in JSON with: intent (login_issue|password_reset|billing|known_issue|general|escalation), "
            "confidence (0-1), agents (list from [KnowledgeAgent, DiagnosticAgent, EscalationAgent]), reasoning (one sentence).\n"
            "If customer asks for human or 2+ failed attempts, include EscalationAgent. For login issues, use both Knowledge + Diagnostic. "
            "Respond with valid JSON only."
        ),
    )

    def __init__(self, llm):
        self.llm = llm

    def route(self, query, history, attempt_count):
        prompt = self.ROUTING_PROMPT.format(query=query, history="\n".join(history[-6:]) if history else "(none)", attempt_count=attempt_count)
        response = self.llm.invoke(prompt)
        try:
            return json.loads(response.content)
        except json.JSONDecodeError:
            return {"intent": "general", "confidence": 0.5, "agents": ["KnowledgeAgent"], "reasoning": "Fallback."}


class KnowledgeAgent:
    QA_PROMPT = PromptTemplate(
        input_variables=["context", "question", "history"],
        template=(
            "You are a helpful support agent. Use the retrieved articles to answer.\n\n"
            "Articles:\n{context}\n\nHistory:\n{history}\n\nQuestion: {question}\n\n"
            "Don't repeat steps the customer already tried. Be concise and empathetic."
        ),
    )

    def __init__(self, llm):
        self.llm = llm
        pass

    def query(self, question, history):
        docs_text = search_knowledge(question)
        context = "\n\n---\n\n".join(docs_text)
        prompt = self.QA_PROMPT.format(context=context, question=question, history="\n".join(history[-6:]) if history else "(none)")
        response = self.llm.invoke(prompt)
        return {"answer": response.content, "sources": ["KB"]}


class DiagnosticAgent:
    def __init__(self, system_health, tickets):
        self.system_health = system_health
        self.tickets = tickets

    def run_diagnostics(self, intent):
        unhealthy = {n: i for n, i in self.system_health.items() if i["status"] != "healthy"}
        keyword_map = {"login_issue": ["log in", "login", "MFA"], "password_reset": ["password", "reset"], "billing": ["billing", "invoice"]}
        keywords = keyword_map.get(intent, [])
        matching = [t for t in self.tickets if t["status"] in ("open", "in_progress") and any(k.lower() in t["subject"].lower() for k in keywords)]
        return {"system_health": "all_clear" if not unhealthy else unhealthy, "known_incidents": matching or "none", "timestamp": datetime.now().isoformat()}


class EscalationAgent:
    def __init__(self, llm):
        self.llm = llm

    def escalate(self, history, diagnostics, sources):
        summary = self.llm.invoke("Summarize this support conversation in 2-3 sentences:\n" + "\n".join(history)).content
        esc_id = f"ESC-{uuid.uuid4().hex[:4].upper()}"
        return {"escalation_id": esc_id, "priority": "high", "summary": summary, "diagnostics": diagnostics, "sources": list(set(sources))}


print("Specialist agents defined: PlannerAgent, KnowledgeAgent, DiagnosticAgent, EscalationAgent")

Specialist agents defined: PlannerAgent, KnowledgeAgent, DiagnosticAgent, EscalationAgent


In [5]:
# Agent Orchestrator

@dataclass
class Session:
    session_id: str = field(default_factory=lambda: f"sess_{uuid.uuid4().hex[:8]}")
    history: list[str] = field(default_factory=list)
    attempt_count: int = 0
    diagnostics: Optional[dict] = None
    sources: list[str] = field(default_factory=list)
    escalated: bool = False


class AgentOrchestrator:
    def __init__(self, planner, knowledge, diagnostic, escalation):
        self.planner = planner
        self.knowledge = knowledge
        self.diagnostic = diagnostic
        self.escalation = escalation
        self.sessions: dict[str, Session] = {}

    def get_or_create_session(self, session_id=None):
        if session_id and session_id in self.sessions:
            return self.sessions[session_id]
        session = Session()
        self.sessions[session.session_id] = session
        return session

    def handle_message(self, message, session_id=None):
        session = self.get_or_create_session(session_id)
        session.history.append(f"Customer: {message}")
        session.attempt_count += 1

        routing = self.planner.route(message, session.history, session.attempt_count)
        print(f"  [Planner] intent={routing['intent']}, agents={routing['agents']}")

        response_parts = []
        for agent_name in routing["agents"]:
            if agent_name == "KnowledgeAgent":
                result = self.knowledge.query(message, session.history)
                session.sources.extend(result["sources"])
                response_parts.append(result["answer"])
                print(f"  [Knowledge] sources: {result['sources']}")
            elif agent_name == "DiagnosticAgent":
                diag = self.diagnostic.run_diagnostics(routing["intent"])
                session.diagnostics = diag
                print(f"  [Diagnostic] health={diag['system_health']}")
            elif agent_name == "EscalationAgent":
                esc = self.escalation.escalate(session.history, session.diagnostics, session.sources)
                session.escalated = True
                response_parts.append(
                    f"I'm connecting you with a support specialist now. "
                    f"Full context has been prepared. Reference: {esc['escalation_id']}."
                )
                print(f"  [Escalation] id={esc['escalation_id']}")

        response = "\n\n".join(response_parts) or "Let me connect you with a specialist."
        session.history.append(f"Agent: {response}")
        return response, session.session_id

print("AgentOrchestrator defined.")

AgentOrchestrator defined.


In [6]:
# Wire Up and Run

planner = PlannerAgent(llm)
knowledge = KnowledgeAgent(llm)
diagnostic = DiagnosticAgent(SYSTEM_HEALTH, SUPPORT_TICKETS)
escalation = EscalationAgent(llm)
orchestrator = AgentOrchestrator(planner, knowledge, diagnostic, escalation)

print("Orchestrator ready.")

Orchestrator ready.


In [7]:
# Demo Conversations

def demo(label, message, session_id=None):
    print(f"\n{'='*70}")
    print(f"  {label}")
    print(f"{'='*70}")
    print(f"\nCustomer: {message}")
    response, sid = orchestrator.handle_message(message, session_id)
    print(f"\nAgent: {response}")
    print(f"[Session: {sid}]")
    return sid

# Login failure — Knowledge + Diagnostic
sid = demo("Scenario 1: Login Failure", "My application fails every time I try to log in")

# Follow-up in same session — context maintained
sid = demo("Scenario 2: Follow-up (same session)", "I tried clearing cache but it still doesn't work", session_id=sid)

# Escalation — frustration + third attempt
sid = demo("Scenario 3: Escalation", "This issue is still not resolved. I need to speak with someone.", session_id=sid)

# New session — simple retrieval
demo("Scenario 4: Password Reset (new session)", "How do I reset my password?")

print(f"\n{'='*70}\nDemo complete.")


  Scenario 1: Login Failure

Customer: My application fails every time I try to log in
  [Planner] intent=general, agents=['KnowledgeAgent']
  [Knowledge] sources: ['KB']

Agent: I'm sorry to hear you're having trouble logging in. Let's try a few common solutions to get you back into your application:

1.  **Clear your browser's cache and cookies.**
2.  Try logging in using an **incognito or private window**.
3.  **Disable any browser extensions** you have active.
4.  **Try a different browser** entirely (e.g., Chrome, especially if you're using Safari 18, as there's a known issue).
5.  Ensure you're using the correct login URL: **app.example.com/login**.
6.  Check your **Caps Lock** key.
7.  If you see an 'Account Locked' message, please **wait 30 minutes** before trying again.

If these steps don't resolve the issue, please let me know, and we can explore further options like checking your network connection or browser console for errors.
[Session: sess_239f1705]

  Scenario 2: Foll